# Simulation 2: cyclic decomposition with ICA post-processing

An 8-edge cyclic graph over 4 source roles and 4 sink roles is laid out as a
4x4 grid. PathFinder fits the 8 observed cells jointly and predicts the 8
missing cells. The per-factor mixing ambiguity is resolved post-hoc by
running ICA on the concatenated row factors.

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from sklearn.decomposition import FastICA

sys.path.insert(0, os.path.abspath('..'))
from pathfinder.decomp import JointOuterDecomp

sns.set_theme(style='whitegrid', context='paper', font_scale=1.15,
              rc={'axes.edgecolor': '.15', 'grid.color': '.88'})
plt.rcParams.update({'font.size': 18})

n_rows = [60, 80, 100, 70]
n_cols = [50, 90, 70, 60]
true_rank = 5
n_comp = 5
noise_level = 0.01
n_restarts = 10
n_iter = 200

obs_pairs = [(0,0), (0,3), (1,0), (1,1), (2,1), (2,2), (3,2), (3,3)]
all_pairs = [(i, j) for i in range(4) for j in range(4)]
miss_pairs = [p for p in all_pairs if p not in obs_pairs]

obs_colour, miss_colour = '#4e79a7', '#f28e2b'
FIGW = FIGH = 7

## Generate structured ground truth

Each factor has block/cluster structure: rows are partitioned into `rank`
contiguous segments, with each component supported on one segment. This gives
non-Gaussian loadings for ICA to latch onto.

In [ ]:
def make_structured_factor(n, rank, rng):
    M = np.zeros((n, rank))
    cuts = np.sort(rng.choice(np.arange(n // 5, n - n // 5),
                              size=rank - 1, replace=False))
    groups = np.split(np.arange(n), cuts)
    for k in range(rank):
        M[groups[k], k] = rng.uniform(0.8, 1.2, size=len(groups[k])) * rng.choice([-1, 1])
        M[:, k] += rng.standard_normal(n) * 0.05
    return M


rng = np.random.default_rng(42)
A_true = [make_structured_factor(nr, true_rank, rng) for nr in n_rows]
S_true = [make_structured_factor(nc, true_rank, rng) for nc in n_cols]

X_clean, X_noisy = {}, {}
for i in range(4):
    for j in range(4):
        C = A_true[i] @ S_true[j].T
        X_clean[(i, j)] = C
        sig = np.linalg.norm(C, 'fro')
        noise = rng.standard_normal(C.shape)
        noise = noise / np.linalg.norm(noise, 'fro') * sig * noise_level
        X_noisy[(i, j)] = C + noise

## Fit PathFinder (multi-restart)

In [ ]:
Clist = [X_noisy[p] for p in obs_pairs]
alpha = [p[0] for p in obs_pairs]
beta = [p[1] for p in obs_pairs]

best_loss = np.inf
best_decomp = None
for restart in range(n_restarts):
    np.random.seed(int(rng.integers(1e9)) + restart * 7)
    d = JointOuterDecomp(n_components=n_comp, n_iter=n_iter, alpha=1e-5,
                         batch_size=10, init_type='random',
                         update_fraction=1.0, verbose=False)
    d.fit(Clist, alpha, beta)
    final_loss = np.mean(d._loss[-1])
    if final_loss < best_loss:
        best_loss, best_decomp = final_loss, d

print(f'Best loss: {best_loss:.6f}')

## ICA post-processing

PathFinder identifies the column space of each $A_i$ / $S_j$ but not the
individual components: any invertible mixing $M$ with
$A_i \leftarrow A_i M$, $S_j \leftarrow S_j M^{-T}$ preserves every product.
We run ICA on the concatenated $A$ factors to find $M$ that maximises
non-Gaussianity, then apply it to every $A_i$ and $M^{-1}$ to every $S_j$.

In [ ]:
A_concat = np.concatenate(best_decomp._A, axis=0)
ica = FastICA(n_components=n_comp, whiten='unit-variance',
              max_iter=5000, tol=1e-6, random_state=0)
ica.fit(A_concat)

A_ica = [ica.transform(A) for A in best_decomp._A]
M_inv = np.linalg.inv(ica.components_)
S_ica = [S @ M_inv for S in best_decomp._S]


def match_factors(true, rec):
    """Greedy column matching by absolute correlation, with sign flips."""
    K = true.shape[1]
    corr = np.array([[np.corrcoef(true[:, i], rec[:, j])[0, 1]
                      for j in range(K)] for i in range(K)])
    abs_corr = np.abs(corr)
    perm, signs, used = [], [], set()
    for i in range(K):
        cand = [(abs_corr[i, j], j) for j in range(K) if j not in used]
        _, j = max(cand)
        perm.append(j); signs.append(np.sign(corr[i, j])); used.add(j)
    return rec[:, perm] * np.array(signs)[None, :]


A_matched = [match_factors(A_true[p], A_ica[p]) for p in range(4)]
S_matched = [match_factors(S_true[q], S_ica[q]) for q in range(4)]

## Reconstruct all 16 cells

In [ ]:
preds = {}
pred_list = best_decomp.predict()
for idx, p in enumerate(obs_pairs):
    preds[p] = pred_list[idx]
for (i, j) in miss_pairs:
    preds[(i, j)] = best_decomp._A[i] @ best_decomp._S[j].T


def compute_r2(true, pred):
    return np.corrcoef(true.flatten(), pred.flatten())[0, 1] ** 2


vlims = {(i, j): max(np.abs(X_clean[(i, j)]).max(),
                      np.abs(preds[(i, j)]).max())
         for i in range(4) for j in range(4)}

## Panel A: configuration grid

Observed cells in blue, missing cells in grey.

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(FIGW, FIGH))
fig.subplots_adjust(hspace=0.2, wspace=0.2)
for i in range(4):
    for j in range(4):
        ax = axes[i, j]
        is_obs = (i, j) in obs_pairs
        fc = obs_colour if is_obs else '#e8e8e8'
        alpha_val = 0.85 if is_obs else 0.40
        rect = patches.FancyBboxPatch(
            (0.05, 0.05), 0.9, 0.9, boxstyle='round,pad=0.04',
            transform=ax.transAxes, facecolor=fc, edgecolor='white',
            linewidth=2.5, alpha=alpha_val)
        ax.add_patch(rect)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        for s in ax.spines.values():
            s.set_visible(False)
        if i == 0:
            ax.set_title(f'S{j}', fontsize=16, fontweight='bold')
        if j == 0:
            ax.set_ylabel(f'A{i}', fontsize=16, fontweight='bold', labelpad=10)
plt.show()

## Panel B: ground truth heatmaps (4x4)

Cell borders: blue = observed, orange = missing.

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(FIGW, FIGH))
fig.subplots_adjust(hspace=0.15, wspace=0.15)
for i in range(4):
    for j in range(4):
        ax = axes[i, j]
        vmax = vlims[(i, j)]
        ax.imshow(X_clean[(i, j)], aspect='auto', cmap='RdBu_r',
                  vmin=-vmax, vmax=vmax, interpolation='nearest')
        ax.set_xticks([]); ax.set_yticks([])
        edge = obs_colour if (i, j) in obs_pairs else miss_colour
        for s in ax.spines.values():
            s.set_edgecolor(edge); s.set_linewidth(2.5)
        if i == 0:
            ax.set_title(f'S{j}', fontsize=10, fontweight='bold')
        if j == 0:
            ax.set_ylabel(f'A{i}', fontsize=10, fontweight='bold', labelpad=5)
fig.suptitle('Ground truth', fontsize=13, fontweight='bold', y=1.02)
plt.show()

## Panel C: true vs reconstructed (observed cells)

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
fig.subplots_adjust(hspace=0.25, wspace=0.25)
for c, (i, j) in enumerate(obs_pairs):
    vmax = vlims[(i, j)]
    for row, mat in enumerate([X_clean[(i, j)], preds[(i, j)]]):
        ax = axes[row, c]
        ax.imshow(mat, aspect='auto', cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                  interpolation='nearest')
        ax.set_xticks([]); ax.set_yticks([])
    axes[0, c].set_title(f'({i},{j})', fontsize=16, fontweight='bold')
axes[0, 0].set_ylabel('True', fontsize=16, fontweight='bold')
axes[1, 0].set_ylabel('Reconstructed', fontsize=16, fontweight='bold')
plt.show()

## Panel D: scatter plots (true vs predicted, all 16 cells)

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(FIGW, FIGH))
fig.subplots_adjust(hspace=0.15, wspace=0.15)
rng_plot = np.random.default_rng(0)
for i in range(4):
    for j in range(4):
        ax = axes[i, j]
        is_obs = (i, j) in obs_pairs
        colour = obs_colour if is_obs else miss_colour
        tf = X_clean[(i, j)].flatten()
        pf = preds[(i, j)].flatten()
        if len(tf) > 3000:
            idx = rng_plot.choice(len(tf), 3000, replace=False)
            tf, pf = tf[idx], pf[idx]
        ax.scatter(tf, pf, s=1.5, alpha=0.25, color=colour,
                   edgecolors='none', rasterized=True)
        lim = max(np.abs(tf).max(), np.abs(pf).max()) * 1.05
        ax.plot([-lim, lim], [-lim, lim], '-', color='#999999',
                linewidth=0.6, zorder=0)
        ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect('equal')
        r2 = np.corrcoef(tf, pf)[0, 1] ** 2
        ax.text(0.05, 0.95, f'R\u00b2={r2:.3f}', transform=ax.transAxes,
                fontsize=6, va='top',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
                          alpha=0.8, edgecolor='#cccccc', linewidth=0.5))
        ax.set_xticks([]); ax.set_yticks([])
        if i == 0:
            ax.set_title(f'S{j}', fontsize=10, fontweight='bold')
        if j == 0:
            ax.set_ylabel(f'A{i}', fontsize=10, fontweight='bold', labelpad=5)
plt.show()

## Panels E and F: subspace and cosine-similarity recovery

Column spaces (E) and row-wise cosine similarity (F) are invariant to the
mixing ambiguity, so they can be compared to ground truth without ICA.

In [ ]:
def proj_matrix(M):
    Q, _ = np.linalg.qr(M, mode='reduced')
    return Q @ Q.T


def cosine_sim(M):
    norms = np.maximum(np.linalg.norm(M, axis=1, keepdims=True), 1e-12)
    Mn = M / norms
    return Mn @ Mn.T


def plot_pair_grid(true_mats, rec_mats, labels, vsym, title):
    fig, axes = plt.subplots(2, len(labels), figsize=(14, 4))
    fig.subplots_adjust(hspace=0.25, wspace=0.25)
    for col, (gt, rc, lab) in enumerate(zip(true_mats, rec_mats, labels)):
        if vsym is None:
            vmax = max(np.abs(gt).max(), np.abs(rc).max())
        else:
            vmax = vsym
        axes[0, col].imshow(gt, cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                             aspect='equal', interpolation='nearest')
        axes[1, col].imshow(rc, cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                             aspect='equal', interpolation='nearest')
        r2 = compute_r2(gt, rc)
        axes[1, col].text(0.05, 0.95, f'R\u00b2={r2:.3f}',
                          transform=axes[1, col].transAxes, fontsize=6, va='top',
                          bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
                                    alpha=0.8, edgecolor='#cccccc', linewidth=0.5))
        axes[0, col].set_title(lab, fontsize=9, fontweight='bold')
        for ax in axes[:, col]:
            ax.set_xticks([]); ax.set_yticks([])
    axes[0, 0].set_ylabel('True', fontsize=10, fontweight='bold')
    axes[1, 0].set_ylabel('Recovered', fontsize=10, fontweight='bold')
    fig.suptitle(title, fontsize=12, fontweight='bold', y=1.03)
    plt.show()


labels_e = [f'P(A{p})' for p in range(4)] + [f'P(S{q})' for q in range(4)]
proj_true = [proj_matrix(A) for A in A_true] + [proj_matrix(S) for S in S_true]
proj_rec = [proj_matrix(A) for A in best_decomp._A] + \
           [proj_matrix(S) for S in best_decomp._S]
plot_pair_grid(proj_true, proj_rec, labels_e, None, 'Subspace projections')

labels_f = [f'A{p}' for p in range(4)] + [f'S{q}' for q in range(4)]
cos_true = [cosine_sim(A) for A in A_true] + [cosine_sim(S) for S in S_true]
cos_rec = [cosine_sim(A) for A in best_decomp._A] + \
          [cosine_sim(S) for S in best_decomp._S]
plot_pair_grid(cos_true, cos_rec, labels_f, 1.0, 'Cosine similarity')

## Panel G: factor recovery with and without ICA

Top: ground truth. Middle: ICA-recovered (matched to truth by correlation).
Bottom: raw PathFinder output without ICA (arbitrary rotation).

In [ ]:
def col_normalise(M):
    norms = np.maximum(np.linalg.norm(M, axis=0, keepdims=True), 1e-12)
    return M / norms


labels = [f'A{p}' for p in range(4)] + [f'S{q}' for q in range(4)]
gt_list = [col_normalise(A) for A in A_true] + [col_normalise(S) for S in S_true]
ic_list = [col_normalise(A) for A in A_matched] + \
          [col_normalise(S) for S in S_matched]
raw_list = [col_normalise(A) for A in best_decomp._A] + \
           [col_normalise(S) for S in best_decomp._S]

fig, axes = plt.subplots(3, 8, figsize=(14, 4))
fig.subplots_adjust(hspace=0.25, wspace=0.25)
for col in range(8):
    gt, ic, raw = gt_list[col], ic_list[col], raw_list[col]
    vmax = max(np.abs(gt).max(), np.abs(ic).max())
    for row, mat in enumerate([gt, ic, raw]):
        ax = axes[row, col]
        ax.imshow(mat, aspect='auto', cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                  interpolation='nearest')
        ax.set_xticks([]); ax.set_yticks([])
    axes[0, col].set_title(labels[col], fontsize=14, fontweight='bold')
axes[0, 0].set_ylabel('True', fontsize=14, fontweight='bold')
axes[1, 0].set_ylabel('Recon', fontsize=14, fontweight='bold')
axes[2, 0].set_ylabel('Recon\n(no ICA)', fontsize=14, fontweight='bold')
plt.show()